# 00 — Normativa Italiana: Appalti Pubblici

Estrae il dataset degli atti italiani sugli appalti da Normattiva,
compatibile con la pipeline del progetto (notebook 02→04).

## Fonte dati

**Normattiva** (`normattiva.it`) — unica fonte. Espone HTML con classi
**Akoma Ntoso (AKN)**, standard internazionale per documenti legislativi.
L'endpoint `~xml` non funziona (risponde sempre con HTML).

## Strategia di fetch

```
1. HTML Normattiva → parser AKN  (articoli/commi/citazioni strutturati)
2. Se 0 articoli  → fallback PDF + pdfplumber + regex
```

## Output

```
data/output/appalti_it/
├── nodes_it.csv            ← metadati (compatibile con nodes_focal.csv)
├── nodes_texts_it.csv      ← + full_text per notebook 03/04
├── edges_it.csv            ← citazioni estratte
├── edges_it_internal.csv   ← solo citazioni seed→seed
├── quality_report.md
└── raw/                    ← HTML e PDF scaricati (cache)
```

## 0. Setup

In [2]:
import re
import time
import json
import requests
import pandas as pd
import pdfplumber
from pathlib import Path
from bs4 import BeautifulSoup
from IPython.display import display, Markdown

OUT_DIR = Path('..') / 'data' / 'output' / 'appalti_it'
RAW_DIR = OUT_DIR / 'raw'
OUT_DIR.mkdir(parents=True, exist_ok=True)
RAW_DIR.mkdir(parents=True, exist_ok=True)

DELAY       = 2.0
TIMEOUT     = 45
MAX_RETRIES = 3
HEADERS     = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36',
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8',
    'Accept-Language': 'it-IT,it;q=0.9,en-US;q=0.7,en;q=0.5',
    'Accept-Encoding': 'gzip, deflate, br',
    'Connection': 'keep-alive',
    'Upgrade-Insecure-Requests': '1',
}
NORMATTIVA_N2LS = 'https://www.normattiva.it/uri-res/N2Ls'

print(f'Output: {OUT_DIR.resolve()}')
print('Setup completato.')

Output: C:\Users\claud\Documents\GitHub\eu-law-network-viz\data\output\appalti_it
Setup completato.


## 1. Catalogo Seed

In [3]:
ATTI_SEED = [

    # ── CODICI ────────────────────────────────────────────────────────────────
    {'slug': 'dlgs_36_2023',  'label': 'D.Lgs. 36/2023',
     'tipo': 'decreto.legislativo', 'data': '2023-03-31', 'numero': '36',
     'titolo': 'Codice dei contratti pubblici',
     'urn': 'urn:nir:stato:decreto.legislativo:2023-03-31;36',
     'layer_atteso': 'Codice primario',
     'note': 'Codice vigente — recepisce Dir. 2014/23, 2014/24, 2014/25',
     'eu_celex_collegati': ['32014L0023', '32014L0024', '32014L0025']},

    {'slug': 'dlgs_50_2016',  'label': 'D.Lgs. 50/2016',
     'tipo': 'decreto.legislativo', 'data': '2016-04-18', 'numero': '50',
     'titolo': 'Codice dei contratti pubblici (abrogato)',
     'urn': 'urn:nir:stato:decreto.legislativo:2016-04-18;50',
     'layer_atteso': 'Codice primario — storico',
     'note': 'Vecchio codice — contratti in corso e analisi storica',
     'eu_celex_collegati': ['32014L0023', '32014L0024', '32014L0025']},

    {'slug': 'dlgs_163_2006', 'label': 'D.Lgs. 163/2006',
     'tipo': 'decreto.legislativo', 'data': '2006-04-12', 'numero': '163',
     'titolo': 'Codice De Lise',
     'urn': 'urn:nir:stato:decreto.legislativo:2006-04-12;163',
     'layer_atteso': 'Codice primario — storico',
     'note': 'Primo codice organico — analisi stratificazione normativa',
     'eu_celex_collegati': ['32004L0017', '32004L0018']},

    # ── CORRETTIVI ────────────────────────────────────────────────────────────
    {'slug': 'dlgs_209_2024', 'label': 'D.Lgs. 209/2024',
     'tipo': 'decreto.legislativo', 'data': '2024-12-31', 'numero': '209',
     'titolo': 'Correttivo al Codice dei contratti pubblici',
     'urn': 'urn:nir:stato:decreto.legislativo:2024-12-31;209',
     'layer_atteso': 'Correttivo',
     'note': 'Correttivo al D.Lgs. 36/2023 — dicembre 2024',
     'eu_celex_collegati': []},

    {'slug': 'dlgs_56_2017',  'label': 'D.Lgs. 56/2017',
     'tipo': 'decreto.legislativo', 'data': '2017-04-19', 'numero': '56',
     'titolo': 'Correttivo al D.Lgs. 50/2016',
     'urn': 'urn:nir:stato:decreto.legislativo:2017-04-19;56',
     'layer_atteso': 'Correttivo',
     'note': 'Primo correttivo al vecchio codice',
     'eu_celex_collegati': []},

    # ── LEGGI DELEGA ──────────────────────────────────────────────────────────
    {'slug': 'l_78_2022',     'label': 'L. 78/2022',
     'tipo': 'legge', 'data': '2022-06-21', 'numero': '78',
     'titolo': 'Delega al Governo in materia di contratti pubblici',
     'urn': 'urn:nir:stato:legge:2022-06-21;78',
     'layer_atteso': 'Legge delega',
     'note': 'Delega che ha abilitato il D.Lgs. 36/2023',
     'eu_celex_collegati': []},

    {'slug': 'l_11_2016',     'label': 'L. 11/2016',
     'tipo': 'legge', 'data': '2016-01-28', 'numero': '11',
     'titolo': 'Delega per recepimento Dir. 2014/23, 2014/24, 2014/25',
     'urn': 'urn:nir:stato:legge:2016-01-28;11',
     'layer_atteso': 'Legge delega',
     'note': 'Delega che ha abilitato il D.Lgs. 50/2016',
     'eu_celex_collegati': []},

    # ── REGOLAMENTO ATTUATIVO ─────────────────────────────────────────────────
    {'slug': 'dpr_207_2010',  'label': 'D.P.R. 207/2010',
     'tipo': 'decreto.del.presidente.della.repubblica',
     'data': '2010-10-05', 'numero': '207',
     'titolo': 'Regolamento di esecuzione del D.Lgs. 163/2006',
     'urn': 'urn:nir:stato:decreto.del.presidente.della.repubblica:2010-10-05;207',
     'layer_atteso': 'Regolamento attuativo',
     'note': 'Archetipo livello III: puro dettaglio operativo, zero principi',
     'eu_celex_collegati': []},

    # ── CANDIDATI IBRIDITÀ ALTA ───────────────────────────────────────────────
    {'slug': 'l_108_2021',    'label': 'L. 108/2021',
     'tipo': 'legge', 'data': '2021-07-29', 'numero': '108',
     'titolo': 'Governance PNRR e semplificazione (conv. D.L. 77/2021)',
     'urn': 'urn:nir:stato:legge:2021-07-29;108',
     'layer_atteso': 'Ibrido atteso',
     'note': 'PNRR: principi e soglie numeriche operative nello stesso articolo',
     'eu_celex_collegati': []},

    {'slug': 'l_120_2020',    'label': 'L. 120/2020',
     'tipo': 'legge', 'data': '2020-09-11', 'numero': '120',
     'titolo': 'Semplificazione e innovazione digitale (conv. D.L. 76/2020)',
     'urn': 'urn:nir:stato:legge:2020-09-11;120',
     'layer_atteso': 'Ibrido atteso',
     'note': 'Misure urgenti semplificazione appalti: mescola soglie, procedure e principi',
     'eu_celex_collegati': []},

    {'slug': 'l_55_2019',     'label': 'L. 55/2019',
     'tipo': 'legge', 'data': '2019-06-14', 'numero': '55',
     'titolo': 'Sblocca Cantieri (conv. D.L. 32/2019)',
     'urn': 'urn:nir:stato:legge:2019-06-14;55',
     'layer_atteso': 'Ibrido atteso',
     'note': 'Deroghe temporanee al codice mescolate con norme di principio',
     'eu_celex_collegati': []},

    {'slug': 'l_114_2014',    'label': 'L. 114/2014',
     'tipo': 'legge', 'data': '2014-08-11', 'numero': '114',
     'titolo': 'Semplificazione e trasparenza amministrativa (conv. D.L. 90/2014)',
     'urn': 'urn:nir:stato:legge:2014-08-11;114',
     'layer_atteso': 'Ibrido atteso',
     'note': 'Modifica il codice appalti su più livelli nello stesso testo',
     'eu_celex_collegati': []},

    # ── LEGGI TRASVERSALI E ANTICORRUZIONE ────────────────────────────────────
    {'slug': 'l_241_1990',    'label': 'L. 241/1990',
     'tipo': 'legge', 'data': '1990-08-07', 'numero': '241',
     'titolo': 'Legge sul procedimento amministrativo',
     'urn': 'urn:nir:stato:legge:1990-08-07;241',
     'layer_atteso': 'Principi generali',
     'note': 'Richiamata in quasi ogni articolo dei codici appalti',
     'eu_celex_collegati': []},

    {'slug': 'l_190_2012',    'label': 'L. 190/2012',
     'tipo': 'legge', 'data': '2012-11-06', 'numero': '190',
     'titolo': 'Legge anticorruzione',
     'urn': 'urn:nir:stato:legge:2012-11-06;190',
     'layer_atteso': 'Legge trasversale',
     'note': 'Obblighi di trasparenza anticorruzione negli appalti',
     'eu_celex_collegati': []},

    {'slug': 'l_90_2024',     'label': 'L. 90/2024',
     'tipo': 'legge', 'data': '2024-06-28', 'numero': '90',
     'titolo': 'Disposizioni in materia di cybersicurezza',
     'urn': 'urn:nir:stato:legge:2024-06-28;90',
     'layer_atteso': 'Legge trasversale',
     'note': 'Obblighi cybersicurezza per appalti ICT e infrastrutture critiche',
     'eu_celex_collegati': []},

    {'slug': 'l_136_2010',    'label': 'L. 136/2010',
     'tipo': 'legge', 'data': '2010-08-13', 'numero': '136',
     'titolo': 'Piano straordinario contro le mafie — tracciabilità finanziaria appalti',
     'urn': 'urn:nir:stato:legge:2010-08-13;136',
     'layer_atteso': 'Legge trasversale',
     'note': 'Introduce obbligo tracciabilità flussi finanziari negli appalti pubblici',
     'eu_celex_collegati': []},

    # ── TRASPARENZA PA ────────────────────────────────────────────────────────
    {'slug': 'dlgs_33_2013',  'label': 'D.Lgs. 33/2013',
     'tipo': 'decreto.legislativo', 'data': '2013-03-14', 'numero': '33',
     'titolo': 'Riordino degli obblighi di pubblicità, trasparenza e diffusione di informazioni',
     'urn': 'urn:nir:stato:decreto.legislativo:2013-03-14;33',
     'layer_atteso': 'Legge trasversale',
     'note': 'Disciplina la pubblicazione degli atti di gara e dei contratti pubblici',
     'eu_celex_collegati': []},

    {'slug': 'dlgs_97_2016',  'label': 'D.Lgs. 97/2016',
     'tipo': 'decreto.legislativo', 'data': '2016-05-25', 'numero': '97',
     'titolo': 'Revisione e semplificazione disposizioni trasparenza (riforma Madia)',
     'urn': 'urn:nir:stato:decreto.legislativo:2016-05-25;97',
     'layer_atteso': 'Correttivo',
     'note': 'Correttivo di D.Lgs. 33/2013 e L. 190/2012 — riforma Madia',
     'eu_celex_collegati': []},

    # ── ANTIMAFIA ─────────────────────────────────────────────────────────────
    {'slug': 'dlgs_218_2012', 'label': 'D.Lgs. 218/2012',
     'tipo': 'decreto.legislativo', 'data': '2012-11-15', 'numero': '218',
     'titolo': 'Disposizioni integrative e correttive al codice antimafia',
     'urn': 'urn:nir:stato:decreto.legislativo:2012-11-15;218',
     'layer_atteso': 'Correttivo',
     'note': 'Aggiorna la disciplina della documentazione antimafia negli appalti',
     'eu_celex_collegati': []},

    # ── OPERE PUBBLICHE — MONITORAGGIO E VALUTAZIONE ──────────────────────────
    {'slug': 'dlgs_228_2011', 'label': 'D.Lgs. 228/2011',
     'tipo': 'decreto.legislativo', 'data': '2011-12-29', 'numero': '228',
     'titolo': 'Valutazione degli investimenti relativi ad opere pubbliche',
     'urn': 'urn:nir:stato:decreto.legislativo:2011-12-29;228',
     'layer_atteso': 'Regolamento attuativo',
     'note': 'Introduce procedure di valutazione ex ante dei grandi appalti pubblici',
     'eu_celex_collegati': []},

    {'slug': 'dlgs_229_2011', 'label': 'D.Lgs. 229/2011',
     'tipo': 'decreto.legislativo', 'data': '2011-12-29', 'numero': '229',
     'titolo': 'Monitoraggio sullo stato di attuazione delle opere pubbliche',
     'urn': 'urn:nir:stato:decreto.legislativo:2011-12-29;229',
     'layer_atteso': 'Regolamento attuativo',
     'note': 'Sistema di monitoraggio fisico-finanziario dei contratti pubblici',
     'eu_celex_collegati': []},

    # ── DIRETTIVA SERVIZI ─────────────────────────────────────────────────────
    {'slug': 'dlgs_59_2010',  'label': 'D.Lgs. 59/2010',
     'tipo': 'decreto.legislativo', 'data': '2010-03-26', 'numero': '59',
     'titolo': 'Attuazione Direttiva Servizi 2006/123/CE',
     'urn': 'urn:nir:stato:decreto.legislativo:2010-03-26;59',
     'layer_atteso': 'Codice primario — storico',
     'note': 'Recepisce la Direttiva Bolkestein — rilevante per concessioni di servizi',
     'eu_celex_collegati': ['32006L0123']},
]

# Assign Normattiva URLs
for atto in ATTI_SEED:
    base = NORMATTIVA_N2LS + '?' + atto['urn']
    atto['html_url'] = base
    atto['pdf_url']  = base + '~pdf'

df_seed = pd.DataFrame(ATTI_SEED)
print(f'Atti nel catalogo seed: {len(ATTI_SEED)}')
print()
for layer, grp in df_seed.groupby('layer_atteso'):
    print(f'  {layer}')
    print(f'    \u2192 {", ".join(grp["label"].tolist())}')


Atti nel catalogo seed: 22

  Codice primario
    → D.Lgs. 36/2023
  Codice primario — storico
    → D.Lgs. 50/2016, D.Lgs. 163/2006, D.Lgs. 59/2010
  Correttivo
    → D.Lgs. 209/2024, D.Lgs. 56/2017, D.Lgs. 97/2016, D.Lgs. 218/2012
  Ibrido atteso
    → L. 108/2021, L. 120/2020, L. 55/2019, L. 114/2014
  Legge delega
    → L. 78/2022, L. 11/2016
  Legge trasversale
    → L. 190/2012, L. 90/2024, L. 136/2010, D.Lgs. 33/2013
  Principi generali
    → L. 241/1990
  Regolamento attuativo
    → D.P.R. 207/2010, D.Lgs. 228/2011, D.Lgs. 229/2011


## 2. Funzioni di Fetch

In [4]:
def safe_get(url, retries=MAX_RETRIES):
    """GET con retry esponenziale."""
    for attempt in range(retries):
        try:
            resp = requests.get(url, headers=HEADERS, timeout=TIMEOUT,
                                allow_redirects=True)
            resp.raise_for_status()
            return resp
        except requests.exceptions.RequestException as e:
            wait = DELAY * (2 ** attempt)
            print(f'    retry {attempt+1}/{retries}: {e} — attendo {wait:.0f}s')
            time.sleep(wait)
    return None


def fetch_html(atto):
    """Scarica HTML Normattiva con cache locale in raw/{slug}.html."""
    slug      = atto['slug']
    html_path = RAW_DIR / f'{slug}.html'
    if html_path.exists():
        return html_path.read_text(encoding='utf-8'), 'html_cache'
    resp = safe_get(atto['html_url'])
    time.sleep(DELAY)
    if not resp or resp.status_code != 200 or len(resp.text) < 5000:
        return None, None
    html_path.write_text(resp.text, encoding='utf-8')
    return resp.text, 'html_fresh'


def fetch_pdf(atto):
    """Scarica PDF Normattiva con cache locale in raw/{slug}.pdf."""
    slug     = atto['slug']
    pdf_path = RAW_DIR / f'{slug}_gu.pdf'
    if not pdf_path.exists():
        alt = RAW_DIR / f'{slug}.pdf'
    if alt.exists():
        pdf_path = alt
        return pdf_path, 'gu_pdf_cache'
    resp = safe_get(atto['pdf_url'])
    time.sleep(DELAY)
    if not resp or resp.status_code != 200:
        return None, None
    ct = resp.headers.get('Content-Type', '')
    if 'pdf' in ct or resp.content[:4] == b'%PDF':
        pdf_path.write_bytes(resp.content)
        return pdf_path, 'pdf_fresh'
    return None, None


print('Funzioni di fetch definite.')

Funzioni di fetch definite.


## 3. Fetch

In [5]:
# ══════════════════════════════════════════════════════════════════════════════
# SEZIONE 1 — Cache guard (skip fetch se dati già presenti)
# ══════════════════════════════════════════════════════════════════════════════

def cache_presente(raw_dir):
    html = list(raw_dir.glob('*.html'))
    pdf  = list(raw_dir.glob('*.pdf'))

    n_tot = len(html) + len(pdf)

    print(f'Cache check → {len(html)} HTML, {len(pdf)} PDF')

    return len(html) >= len(ATTI_SEED)


SKIP_FETCH = cache_presente(RAW_DIR)

if SKIP_FETCH:
    print('✓ Cache già presente → salto fetch')
else:
    print('Nessuna cache → eseguo fetch')

Cache check → 22 HTML, 16 PDF
✓ Cache già presente → salto fetch


In [6]:
import re
import time
import json
import fitz          # pymupdf
import requests
import pdfplumber
import pandas as pd
from pathlib import Path
from bs4 import BeautifulSoup
from datetime import date

# ══════════════════════════════════════════════════════════════════════════════
# SEZIONE 2 — Funzioni di fetch
# ══════════════════════════════════════════════════════════════════════════════

NORMATTIVA_BASE = 'https://www.normattiva.it'
DATA_VIGENZA    = date.today().strftime('%Y%m%d')

GU_SKIP = [
    'Gazzetta Ufficiale', 'Spediz. abb', 'ISTITUTO POLIGRAFICO',
    'SI PUBBLICA TUTTI', 'SOMMARIO', 'CENTRALINO',
    'COPIA TRATTA DA GURITEL', 'Supplemento ordinario',
    'Serie generale - n.', 'LEGGI ED ALTRI ATTI',
]


def atto_gia_pronto(slug):
    return (RAW_DIR / f'{slug}.html').exists() and (
        (RAW_DIR / f'{slug}_gu.pdf').exists() or
        (RAW_DIR / f'{slug}.pdf').exists()
    )


def load_cached_result(slug):
    path = RAW_DIR / f'{slug}.json'
    if path.exists():
        with open(path, 'r', encoding='utf-8') as f:
            return json.load(f)
    return None


def save_result(slug, res):
    with open(RAW_DIR / f'{slug}.json', 'w', encoding='utf-8') as f:
        json.dump(res, f, ensure_ascii=False)


def safe_get(url, params=None, retries=MAX_RETRIES):
    for attempt in range(retries):
        try:
            r = requests.get(url, params=params, headers=HEADERS,
                             timeout=TIMEOUT, allow_redirects=True)
            r.raise_for_status()
            return r
        except requests.exceptions.RequestException as e:
            wait = DELAY * (2 ** attempt)
            print(f'    retry {attempt+1}/{retries}: {e} — attendo {wait:.0f}s')
            time.sleep(wait)
    return None


def fetch_html(atto):
    slug      = atto['slug']
    html_path = RAW_DIR / f'{slug}.html'
    if html_path.exists():
        return html_path.read_text(encoding='utf-8', errors='replace'), 'html_cache'
    resp = safe_get(atto['html_url'])
    time.sleep(DELAY)
    if not resp or resp.status_code != 200 or len(resp.text) < 5000:
        return None, None
    html_path.write_text(resp.text, encoding='utf-8')
    return resp.text, 'html_fresh'


def estrai_meta_da_html(html):
    soup = BeautifulSoup(html, 'html.parser')
    meta = {
        'codice_redazionale': '', 'data_gu': '', 'titolo': '',
        'preambolo': '', 'urn_citazioni': [], 'gu_pdf_url': '',
    }

    for inp in soup.find_all('input', {'name': True}):
        name = inp.get('name', '')
        val  = inp.get('value', '')
        if 'codiceRedazionale' in name and val:
            meta['codice_redazionale'] = val
        if 'dataPubblicazioneGazzetta' in name and val:
            meta['data_gu'] = val.replace('-', '')

    if not meta['codice_redazionale']:
        for a in soup.find_all('a', href=True):
            m = re.search(r'codiceRedaz=([A-Z0-9]+)', a['href'])
            if m:
                meta['codice_redazionale'] = m.group(1)
            m2 = re.search(r'dataGU=(\d{8})', a['href'])
            if m2:
                meta['data_gu'] = m2.group(1)

    testa = soup.find(class_='testa_atto')
    meta['titolo'] = testa.get_text(separator=' ', strip=True)[:300] if testa else ''

    preamble_parts = []
    for cls in ('preamble-title-akn', 'preamble-citations-akn',
                 'preamble-text-akn', 'preamble-end-akn', 'formula-introduttiva'):
        for tag in soup.find_all(class_=cls):
            t = tag.get_text(separator=' ', strip=True)
            if t:
                preamble_parts.append(t)
    meta['preambolo'] = ' '.join(preamble_parts)

    seen = set()
    for a in soup.find_all('a', href=True):
        href = a['href']
        m = re.search(r'(urn:nir:[^\s\'"<>&]+)', href)
        if m:
            urn_base = re.sub(r'~.*$', '', m.group(1).rstrip('~/ '))
            if urn_base not in seen:
                seen.add(urn_base)
                meta['urn_citazioni'].append({'urn': urn_base, 'testo': a.get_text(strip=True)})

    for a in soup.find_all('a', href=True):
        if 'gazzettaufficiale.it' in a['href'] and 'pdf' in a['href'].lower():
            meta['gu_pdf_url'] = a['href']
            break

    return meta


def fetch_gu_pdf(gu_pdf_url, slug):
    for candidate in [RAW_DIR / f'{slug}_gu.pdf', RAW_DIR / f'{slug}.pdf']:
        if candidate.exists():
            return candidate, 'gu_pdf_cache'
    if not gu_pdf_url:
        return None, None
    resp = safe_get(gu_pdf_url)
    time.sleep(DELAY)
    if not resp or resp.status_code != 200:
        return None, None
    ct = resp.headers.get('Content-Type', '')
    if 'pdf' in ct or resp.content[:4] == b'%PDF':
        pdf_path = RAW_DIR / f'{slug}_gu.pdf'
        pdf_path.write_bytes(resp.content)
        return pdf_path, 'gu_pdf_fresh'
    return None, None


# ══════════════════════════════════════════════════════════════════════════════
# SEZIONE 3 — Parser
# ══════════════════════════════════════════════════════════════════════════════

RE_ARTICOLO = re.compile(
    r'(?m)^[ \t]*Art(?:icolo)?\.?\s+'
    r'(\d+(?:\s*-?\s*(?:bis|ter|quater|quinquies|sexies|septies|octies|novies|decies))?)'
    r'[ \t]*\.?[ \t]*(?:\(([^)\n]{0,120})\))?[ \t]*$',
    re.IGNORECASE
)
RE_COMMA_COUNT = re.compile(r'(?m)^\s*\d+\.\s+\S')

TIPO_MAP_CIT = [
    (re.compile(r'(?i)\bdecreto\s+legislativo\s+(?:n\.?\s*)?(\d+)\s*/\s*(\d{4})'), 'decreto.legislativo'),
    (re.compile(r'(?i)\bd\.?\s*lgs\.?\s+(?:n\.?\s*)?(\d+)\s*/\s*(\d{4})'),        'decreto.legislativo'),
    (re.compile(r'(?i)\bdecreto(?:\s+del)?\s*presidente\s+della\s+repubblica\s+(?:n\.?\s*)?(\d+)\s*/\s*(\d{4})'), 'decreto.del.presidente.della.repubblica'),
    (re.compile(r'(?i)\bd\.?\s*p\.?\s*r\.?\s+(?:n\.?\s*)?(\d+)\s*/\s*(\d{4})'),  'decreto.del.presidente.della.repubblica'),
    (re.compile(r'(?i)\bdecreto[\s\-]legge\s+(?:n\.?\s*)?(\d+)\s*/\s*(\d{4})'),  'decreto.legge'),
    (re.compile(r'(?i)\bd\.?\s*l\.?\s+(?:n\.?\s*)?(\d+)\s*/\s*(\d{4})'),         'decreto.legge'),
    (re.compile(r'(?i)\blegge\s+(?:n\.?\s*)?(\d+)\s*/\s*(\d{4})'),                'legge'),
    (re.compile(r'(?i)\bl\.\s*(\d+)\s*/\s*(\d{4})'),                              'legge'),
]


def estrai_citazioni_da_testo(testo):
    citazioni, seen_urns = [], set()
    for pat, tipo in TIPO_MAP_CIT:
        for m in pat.finditer(testo):
            try:
                numero, anno = m.group(1).strip(), m.group(2).strip()
            except IndexError:
                continue
            if not anno.isdigit() or not (1948 <= int(anno) <= 2025):
                continue
            urn = f"urn:nir:stato:{tipo}:{anno};{numero}"
            if urn not in seen_urns:
                seen_urns.add(urn)
                citazioni.append({'urn': urn, 'testo': m.group(0).strip()})
    return citazioni


def estrai_articoli_da_testo(full_text):
    """Segmenta full_text in articoli tramite RE_ARTICOLO."""
    splits = list(RE_ARTICOLO.finditer(full_text))
    if len(splits) < 2:
        return []
    articoli = []
    for i, m in enumerate(splits):
        art_num = m.group(1).strip()
        rubrica = (m.group(2) or '').strip()
        start   = m.start()
        end     = splits[i+1].start() if i+1 < len(splits) else len(full_text)
        testo   = full_text[start:end].strip()
        articoli.append({
            'id': f'art{art_num}', 'numero': art_num, 'rubrica': rubrica,
            'testo': testo[:10000],
            'n_commi': max(len(RE_COMMA_COUNT.findall(testo)), 1),
            'n_modifiche': 0,
        })
    return articoli


def estrai_testo_da_pdf(pdf_path):
    """
    Estrae testo da PDF GU usando pymupdf con filtro intestazioni.
    Ricongiunge parole spezzate con trattino a fine riga.
    """
    doc = fitz.open(str(pdf_path))
    pages_text = []
    for page in doc:
        blocks = page.get_text("blocks", sort=True)
        lines = []
        for b in blocks:
            txt = b[4].strip()
            if not txt or len(txt) < 5:
                continue
            if any(p in txt for p in GU_SKIP):
                continue
            lines.append(txt)
        if lines:
            pages_text.append('\n'.join(lines))
    doc.close()
    full = '\n'.join(pages_text)
    # ricongiunge parole spezzate a fine riga (es. "esecu-\ntiva" → "esecutiva")
    full = re.sub(r'-\n\s*', '', full)
    return full


def parse_pdf_atto(pdf_path, slug):
    result = {'slug': slug, 'fonte': 'pdf_gu', 'errore': None}
    try:
        # n_pages da pdfplumber (leggero), testo da pymupdf (accurato)
        with pdfplumber.open(pdf_path) as pdf:
            n_pages = len(pdf.pages)
        full_text = estrai_testo_da_pdf(pdf_path)
    except Exception as e:
        result.update({'errore': str(e), 'n_articoli': 0,
                       'articoli': [], 'citazioni': [], 'n_commi': 0,
                       'n_citazioni': 0, 'full_text': ''})
        return result

    result['n_pagine'] = n_pages

    if not full_text or len(full_text) / max(n_pages, 1) < 80:
        result.update({'errore': 'PDF scansione o vuoto', 'n_articoli': 0,
                       'articoli': [], 'citazioni': [], 'n_commi': 0,
                       'n_citazioni': 0, 'full_text': ''})
        return result

    result['full_text'] = full_text

    articoli = estrai_articoli_da_testo(full_text)
    if articoli:
        result['articoli']   = articoli
        result['n_articoli'] = len(articoli)
        result['n_commi']    = sum(a['n_commi'] for a in articoli)
    else:
        result.update({'errore': '0 articoli trovati con regex',
                       'n_articoli': 0, 'articoli': [], 'n_commi': 0})

    citazioni = estrai_citazioni_da_testo(full_text)
    result['citazioni']   = citazioni
    result['n_citazioni'] = len(citazioni)
    return result


def estrai_articoli_da_html_akn(html, meta, atto):
    """Estrae articoli dalle classi AKN dell'HTML Normattiva."""
    soup = BeautifulSoup(html, 'html.parser')
    art_segs = []
    for num_tag in soup.find_all(class_='article-num-akn'):
        parent = num_tag.find_parent()
        if not parent:
            continue
        num = re.sub(r'[^\d\w]', '', num_tag.get_text(strip=True))
        heading   = parent.find(class_='article-heading-akn')
        rubrica   = heading.get_text(strip=True) if heading else ''
        commi_div = parent.find(class_='art-commi-div-akn')
        testo_raw = (commi_div or parent).get_text(separator=' ', strip=True)
        testo = re.sub(r'\s+', ' ', testo_raw).strip()
        if len(testo) < 20:
            continue
        art_segs.append({
            'id': f'art{num}', 'numero': num, 'rubrica': rubrica,
            'testo': f'Art. {num}. {rubrica}\n{testo}'[:10000],
            'n_commi': 1, 'n_modifiche': 0,
        })
    return art_segs


# ══════════════════════════════════════════════════════════════════════════════
# SEZIONE 4 — Fetch completo per atto
# ══════════════════════════════════════════════════════════════════════════════

def fetch_atto_completo(atto):
    """
    Pipeline per singolo atto:
      1. HTML Normattiva → estrai meta + URN citazioni
      2. Classi AKN dall'HTML → articoli strutturati (atti vigenti)
      3. PDF Gazzetta Ufficiale + pymupdf → fallback (atti abrogati/storici)
      4. Fallimento totale → restituisce almeno le citazioni HTML
    """
    slug  = atto['slug']
    label = atto['label']

    # ── 1. HTML ──────────────────────────────────────────────────────────────
    html, _ = fetch_html(atto)
    meta = estrai_meta_da_html(html) if html else {
        'codice_redazionale': '', 'data_gu': '', 'titolo': '',
        'preambolo': '', 'urn_citazioni': [], 'gu_pdf_url': '',
    }

    # ── 2. AKN HTML ──────────────────────────────────────────────────────────
    if html:
        art_segs = estrai_articoli_da_html_akn(html, meta, atto)
        if len(art_segs) >= 2:
            print(f'  [{label}] ✓ HTML AKN — {len(art_segs)} art, '
                  f'{len(meta["urn_citazioni"])} cit')
            return {
                'slug': slug, 'fonte': 'html_normattiva',
                'fonte_file': f'{slug}.html',
                'titolo_estratto': meta['titolo'] or atto['titolo'],
                'preambolo': meta['preambolo'],
                'articoli': art_segs,
                'n_articoli': len(art_segs),
                'n_commi': len(art_segs),
                'citazioni': meta['urn_citazioni'],
                'n_citazioni': len(meta['urn_citazioni']),
                'full_text': '\n\n'.join(a['testo'] for a in art_segs),
            }

    # ── 3. PDF GU (pymupdf) ──────────────────────────────────────────────────
    pdf_path, pdf_fonte = fetch_gu_pdf(meta.get('gu_pdf_url', ''), slug)
    if pdf_path:
        parsed = parse_pdf_atto(pdf_path, slug)
        parsed['fonte_file']      = pdf_fonte
        parsed['titolo_estratto'] = meta['titolo'] or atto['titolo']
        parsed['preambolo']       = meta['preambolo']

        # unisci citazioni HTML + regex PDF
        seen = {c['urn'] for c in parsed.get('citazioni', [])}
        for c in meta['urn_citazioni']:
            if c['urn'] not in seen:
                seen.add(c['urn'])
                parsed['citazioni'].append(c)
        parsed['n_citazioni'] = len(parsed.get('citazioni', []))

        if parsed.get('errore'):
            print(f'  [{label}] ⚠ PDF GU: {parsed["errore"]}')
        else:
            print(f'  [{label}] ✓ PDF GU — '
                  f'{parsed["n_articoli"]} art, {parsed["n_citazioni"]} cit')
        return parsed

    # ── 4. Fallimento ─────────────────────────────────────────────────────────
    print(f'  [{label}] ✗ FALLITO')
    return {
        'slug': slug, 'fonte': 'nessuna', 'fonte_file': None,
        'errore': 'HTML AKN e PDF GU non disponibili',
        'titolo_estratto': meta['titolo'] or atto['titolo'],
        'preambolo': meta['preambolo'],
        'n_articoli': 0, 'n_commi': 0,
        'citazioni': meta['urn_citazioni'],
        'n_citazioni': len(meta['urn_citazioni']),
        'articoli': [], 'full_text': '',
    }


# ══════════════════════════════════════════════════════════════════════════════
# SEZIONE 5 — Fetch batch
# ══════════════════════════════════════════════════════════════════════════════

print(f'\nFetching {len(ATTI_SEED)} atti...')
print('=' * 60)

results_raw = []

for i, atto in enumerate(ATTI_SEED):
    slug = atto['slug']
    print(f'[{i+1:2d}/{len(ATTI_SEED)}]', end=' ')

    cached = load_cached_result(slug)
    if cached is not None:
        print(f'⏭ cache {atto["label"]} ({cached.get("fonte", "?")})')
        res = cached
    else:
        print(f'↓ fetch {atto["label"]}')
        res = fetch_atto_completo(atto)
        save_result(slug, res)

    for k in ('tipo', 'data', 'numero', 'titolo', 'urn',
               'layer_atteso', 'note', 'eu_celex_collegati'):
        res[k] = atto.get(k, '')

    results_raw.append(res)

print('\nFetch completato.')


Fetching 22 atti...
[ 1/22] ⏭ cache D.Lgs. 36/2023 (pdf_gu)
[ 2/22] ⏭ cache D.Lgs. 50/2016 (pdf_gu)
[ 3/22] ⏭ cache D.Lgs. 163/2006 (pdf_gu)
[ 4/22] ⏭ cache D.Lgs. 209/2024 (pdf_gu)
[ 5/22] ⏭ cache D.Lgs. 56/2017 (pdf_gu)
[ 6/22] ⏭ cache L. 78/2022 (pdf_gu)
[ 7/22] ⏭ cache L. 11/2016 (pdf_gu)
[ 8/22] ⏭ cache D.P.R. 207/2010 (pdf_gu)
[ 9/22] ⏭ cache L. 108/2021 (pdf_gu)
[10/22] ⏭ cache L. 120/2020 (pdf_gu)
[11/22] ⏭ cache L. 55/2019 (pdf_gu)
[12/22] ⏭ cache L. 114/2014 (pdf_gu)
[13/22] ⏭ cache L. 241/1990 (pdf_gu)
[14/22] ⏭ cache L. 190/2012 (pdf_gu)
[15/22] ⏭ cache L. 90/2024 (pdf_gu)
[16/22] ⏭ cache L. 136/2010 (pdf_gu)
[17/22] ↓ fetch D.Lgs. 33/2013
  [D.Lgs. 33/2013] ✓ PDF GU — 109 art, 22 cit
[18/22] ↓ fetch D.Lgs. 97/2016
  [D.Lgs. 97/2016] ✓ PDF GU — 74 art, 19 cit
[19/22] ↓ fetch D.Lgs. 218/2012
  [D.Lgs. 218/2012] ✓ PDF GU — 33 art, 26 cit
[20/22] ↓ fetch D.Lgs. 228/2011
  [D.Lgs. 228/2011] ✓ PDF GU — 37 art, 24 cit
[21/22] ↓ fetch D.Lgs. 229/2011
  [D.Lgs. 229/2011] ✓ PDF GU 

In [7]:
import fitz

def extract_full_text_pymupdf(pdf_path):
    import fitz, re
    doc = fitz.open(str(pdf_path))
    pages_text = []
    SKIP_PATTERNS = [
        'Gazzetta Ufficiale', 'Spediz. abb', 'ISTITUTO POLIGRAFICO',
        'SI PUBBLICA TUTTI', 'SOMMARIO', 'CENTRALINO',
        'COPIA TRATTA DA GURITEL', 'Supplemento ordinario',
        'Serie generale - n.', 'LEGGI ED ALTRI ATTI',
    ]
    for page in doc:
        blocks = page.get_text("blocks", sort=True)
        lines = []
        for b in blocks:
            txt = b[4].strip()
            if not txt or len(txt) < 5:
                continue
            if any(p in txt for p in SKIP_PATTERNS):
                continue
            lines.append(txt)
        if lines:
            pages_text.append('\n'.join(lines))
    doc.close()
    full = '\n'.join(pages_text)
    # ricongiunge parole spezzate con trattino a fine riga
    full = re.sub(r'-\n(\s*)', '', full)
    return full

# test
from pathlib import Path
RAW_DIR = Path('../data/output/appalti_it/raw')
print(repr(extract_full_text_pymupdf(RAW_DIR / 'dlgs_163_2006_gu.pdf')[:1000]))

'N. 107/L\nDECRETO LEGISLATIVO 12 aprile 2006, n. 163.\nCodice dei contratti pubblici relativi a\nlavori, servizi e forniture in attuazione delle\ndirettive 2004/17/CE e 2004/18/CE.\nS O M M A R I O\nöööö\nDECRETO LEGISLATIVO 12 aprile 2006, n. 163. ö Codice dei contratti pubblici\nrelativi a lavori, servizi e forniture in attuazione delle direttive 2004/17/CE e\n2004/18/CE. . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . .\nPag.\n3\nNote . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . .\ný\n233\nö 1 ö\nDECRETO LEGISLATIVO 12 aprile 2006, n. 163.\nE m a n a\nil seguente decreto legislativo :\nParte I\nCodice dei contratti pubblici relativi a lavori, servizi e forniture in attuazione delle direttive 2004/17/CE e 2004/18/CE.\nPRINCIPI E DISPOSIZIONI COMU

## 6. Quality Report

In [8]:
quality_rows = []
for res in results_raw:
    n_art       = res.get('n_articoli', 0)
    n_cit       = res.get('n_citazioni', 0)
    testo_chars = sum(len(a.get('testo', '')) for a in res.get('articoli', []))
    errore      = res.get('errore', '')

    score = 0
    if not errore:             score += 20
    if n_art > 100:            score += 30
    elif n_art > 20:           score += 20
    elif n_art > 5:            score += 10
    elif n_art > 0:            score += 5
    if n_cit > 30:             score += 25
    elif n_cit > 10:           score += 15
    elif n_cit > 0:            score += 5
    if testo_chars > 100_000:  score += 25
    elif testo_chars > 20_000: score += 15
    elif testo_chars > 2_000:  score += 5

    quality_rows.append({
        'slug':          res['slug'],
        'label':         next(a['label'] for a in ATTI_SEED if a['slug'] == res['slug']),
        'layer_atteso':  res.get('layer_atteso', ''),
        'fonte':         res.get('fonte', 'nessuna'),
        'fonte_file':    res.get('fonte_file', ''),
        'n_articoli':    n_art,
        'n_commi':       res.get('n_commi', 0),
        'n_citazioni':   n_cit,
        'testo_chars':   testo_chars,
        'errore':        errore or '',
        'quality_score': score,
    })

df_quality = pd.DataFrame(quality_rows)

print('QUALITY REPORT')
print('=' * 75)
display(df_quality[[
    'label', 'fonte', 'n_articoli', 'n_commi', 'n_citazioni',
    'testo_chars', 'quality_score'
]].to_string(index=False))

print()
print(f"  Con testo:      {(df_quality['n_articoli'] > 0).sum()}/{len(df_quality)}")
print(f"  HTML/AKN:       {(df_quality['fonte'] == 'html_akn').sum()}")
print(f"  PDF:            {(df_quality['fonte'] == 'pdf').sum()}")
print(f"  Falliti:        {(df_quality['fonte'] == 'nessuna').sum()}")
print(f"  Score medio:    {df_quality['quality_score'].mean():.0f}/100")
print(f"  Totale art.:    {df_quality['n_articoli'].sum()}")
print(f"  Totale cit.:    {df_quality['n_citazioni'].sum()}")

problemi = df_quality[df_quality['quality_score'] < 40]
if not problemi.empty:
    print(f"\nAtti con score < 40:")
    for _, r in problemi.iterrows():
        urn = next(a['urn'] for a in ATTI_SEED if a['slug'] == r['slug'])
        print(f"  {r['label']}: {r['errore'] or str(r['n_articoli']) + ' art'}")
        print(f"    → Verifica URN su normattiva.it: {urn}")

QUALITY REPORT


'          label  fonte  n_articoli  n_commi  n_citazioni  testo_chars  quality_score\n D.Lgs. 36/2023 pdf_gu         489     2725            9      1382685             80\n D.Lgs. 50/2016 pdf_gu           0        0           10            0              5\nD.Lgs. 163/2006 pdf_gu         257     1669           41       653820            100\nD.Lgs. 209/2024 pdf_gu           0        0            6            0              5\n D.Lgs. 56/2017 pdf_gu         131      970           11       163495             90\n     L. 78/2022 pdf_gu           7       29           17        40571             60\n     L. 11/2016 pdf_gu          71      157           31       164680             90\nD.P.R. 207/2010 pdf_gu           0        0           12            0             15\n    L. 108/2021 pdf_gu         122      515            4       405440             80\n    L. 120/2020 pdf_gu         112      483            3       443377             80\n     L. 55/2019 pdf_gu         111      397          


  Con testo:      17/22
  HTML/AKN:       0
  PDF:            0
  Falliti:        0
  Score medio:    64/100
  Totale art.:    1930
  Totale cit.:    374

Atti con score < 40:
  D.Lgs. 50/2016: 0 articoli trovati con regex
    → Verifica URN su normattiva.it: urn:nir:stato:decreto.legislativo:2016-04-18;50
  D.Lgs. 209/2024: 0 articoli trovati con regex
    → Verifica URN su normattiva.it: urn:nir:stato:decreto.legislativo:2024-12-31;209
  D.P.R. 207/2010: 0 articoli trovati con regex
    → Verifica URN su normattiva.it: urn:nir:stato:decreto.del.presidente.della.repubblica:2010-10-05;207
  L. 241/1990: PDF scansione o vuoto
    → Verifica URN su normattiva.it: urn:nir:stato:legge:1990-08-07;241
  D.Lgs. 59/2010: 0 articoli trovati con regex
    → Verifica URN su normattiva.it: urn:nir:stato:decreto.legislativo:2010-03-26;59


## 7. Grafo Citazioni

In [9]:
urn_to_slug = {a['urn']: a['slug'] for a in ATTI_SEED}

def normalize_urn(raw):
    m = re.search(r'(urn:nir:[^\s~&"<]+)', raw or '')
    return m.group(1).rstrip('/ ') if m else None


edges = []
for res in results_raw:
    src_slug = res['slug']
    src_urn  = res.get('urn', '')
    for cit in res.get('citazioni', []):
        norm = normalize_urn(cit.get('urn', '')) or normalize_urn(cit.get('testo', ''))
        if not norm or norm == src_urn:
            continue
        edges.append({
            'src_slug':  src_slug,
            'src_urn':   src_urn,
            'dst_urn':   norm,
            'dst_slug':  urn_to_slug.get(norm),
            'testo_rif': cit.get('testo', '')[:200],
            'interno':   urn_to_slug.get(norm) is not None,
        })

df_edges = (pd.DataFrame(edges) if edges
            else pd.DataFrame(columns=['src_slug','src_urn','dst_urn',
                                       'dst_slug','testo_rif','interno']))

n_int = df_edges['interno'].sum() if len(df_edges) else 0
print(f"Citazioni totali:      {len(df_edges)}")
print(f"Interne (seed→seed):   {n_int}")

if len(df_edges) > 0:
    print(f"\nCitazioni per atto fonte:")
    print(df_edges.groupby('src_slug').size().sort_values(ascending=False).to_string())
    interni = df_edges[df_edges['interno']]
    if len(interni) > 0:
        print(f"\nAtti più citati (interni):")
        print(interni.groupby('dst_slug').size().sort_values(ascending=False).to_string())

Citazioni totali:      372
Interne (seed→seed):   33

Citazioni per atto fonte:
src_slug
l_190_2012       72
dlgs_163_2006    41
l_11_2016        31
dlgs_218_2012    26
dlgs_228_2011    24
dlgs_33_2013     22
l_136_2010       21
dlgs_97_2016     19
l_78_2022        17
l_90_2024        13
dlgs_229_2011    13
dpr_207_2010     12
dlgs_56_2017     11
dlgs_50_2016     10
l_55_2019        10
dlgs_36_2023      9
dlgs_209_2024     6
dlgs_59_2010      5
l_108_2021        4
l_120_2020        3
l_114_2014        3

Atti più citati (interni):
dst_slug
dlgs_163_2006    8
dlgs_50_2016     5
dlgs_33_2013     3
dlgs_36_2023     3
l_241_1990       3
l_136_2010       2
l_78_2022        2
l_190_2012       2
dlgs_228_2011    1
l_11_2016        1
l_114_2014       1
dlgs_97_2016     1
dlgs_56_2017     1


## 8. Export CSV + Quality Report

In [10]:
nodes_rows = []
for res in results_raw:
    slug  = res['slug']
    seed  = next(a for a in ATTI_SEED if a['slug'] == slug)
    score = next(r['quality_score'] for r in quality_rows if r['slug'] == slug)

    # Ricostruisce full_text
    if res.get('full_text'):          # percorso PDF
        full_text = res['full_text']
    else:                             # percorso HTML/AKN
        parts = [f"TITOLO: {res.get('titolo_estratto') or seed['titolo']}"]
        if res.get('preambolo'):
            parts.append(f"\nPREAMBOLO:\n{res['preambolo']}")
        for art in res.get('articoli', []):
            rubrica = f" — {art['rubrica']}" if art.get('rubrica') else ''
            parts.append(f"\nArticolo {art['numero']}{rubrica}\n{art['testo']}")
        full_text = '\n'.join(parts)

    nodes_rows.append({
        'id':                 slug,
        'urn':                seed['urn'],
        'giurisdizione':      'IT',
        'tipo':               seed['tipo'],
        'anno':               int(seed['data'][:4]),
        'numero':             seed['numero'],
        'titolo':             res.get('titolo_estratto') or seed['titolo'],
        'layer_atteso':       seed['layer_atteso'],
        'eu_celex_collegati': json.dumps(seed.get('eu_celex_collegati', [])),
        'fonte_estrazione':   res.get('fonte', 'nessuna'),
        'fonte_file':         res.get('fonte_file', ''),
        'n_articoli':         res.get('n_articoli', 0),
        'n_commi':            res.get('n_commi', 0),
        'quality_score':      score,
        'note':               seed['note'],
        'full_text':          full_text,
    })

df_nodes = pd.DataFrame(nodes_rows)

nodes_path = OUT_DIR / 'nodes_it.csv'
df_nodes.drop(columns=['full_text']).to_csv(nodes_path, index=False, encoding='utf-8')
print(f'✓ {nodes_path}')

texts_path = OUT_DIR / 'nodes_texts_it.csv'
df_nodes.to_csv(texts_path, index=False, encoding='utf-8')
print(f'✓ {texts_path} (con full_text)')

df_edges.to_csv(OUT_DIR / 'edges_it.csv', index=False, encoding='utf-8')
print(f'✓ edges_it.csv ({len(df_edges)} archi)')

if len(df_edges) > 0:
    df_int = df_edges[df_edges['interno']]
    df_int.to_csv(OUT_DIR / 'edges_it_internal.csv', index=False, encoding='utf-8')
    print(f'✓ edges_it_internal.csv ({len(df_int)} archi interni)')

✓ ..\data\output\appalti_it\nodes_it.csv
✓ ..\data\output\appalti_it\nodes_texts_it.csv (con full_text)
✓ edges_it.csv (372 archi)
✓ edges_it_internal.csv (33 archi interni)


## 9. Integrazione EUR-Lex — Direttive europee sugli appalti

Recupera il testo integrale delle direttive EU collegate agli atti seed italiani.
I CELEX vengono estratti automaticamente da `eu_celex_collegati` — nessuna lista
manuale da mantenere. Output: `nodes_texts_eu_appalti.csv` con le stesse colonne
di `nodes_texts_it.csv`, più `giurisdizione='EU'`.

Il 04_layer_inference_ita_law.ipynb carica e concatena i due CSV prima del
clustering — le direttive ancoreranno i cluster a livello framework/principi.

In [11]:
# ── 9.1  Import e costanti  ──────────────────────────────────────────────────
import re, time, json
import pandas as pd
from pathlib import Path
from bs4 import BeautifulSoup
import eurlex  

# ── Carica CELEX da nodes_eu_appalti.csv ────────────────────────────────
eu_nodes_path = OUT_DIR / 'nodes_eu_appalti.csv'

df_nodes_eu = pd.read_csv(eu_nodes_path)

# colonna corretta = 'celex'
celex_ids = (
    df_nodes_eu['celex']
    .dropna()
    .astype(str)
    .str.strip()
    .unique()
    .tolist()
)

print(f"CELEX da recuperare: {len(celex_ids)}")

# ── 9.2  Funzioni di estrazione (identiche al notebook 03) ───────────────────

MAX_TITLE_CHARS = 500
PREAMBLE_END_MARKERS = [
    'HAVE ADOPTED THIS REGULATION:', 'HAS ADOPTED THIS REGULATION:',
    'HAVE ADOPTED THIS DIRECTIVE:',  'HAS ADOPTED THIS DIRECTIVE:',
    'HAVE ADOPTED THIS DECISION:',   'HAS ADOPTED THIS DECISION:',
    'HAVE ADOPTED THIS FRAMEWORK DECISION:', 'HAVE ADOPTED THIS RECOMMENDATION:',
    'HEREBY DECIDES:', 'HAS DECIDED AS FOLLOWS:', 'HEREBY RECOMMENDS:',
    'IS OF THE OPINION THAT:', 'HAVE AGREED AS FOLLOWS:', 'HAVE DECIDED AS FOLLOWS:',
]


def fetch_eurlex_html_lib(celex: str):
    """Usa la libreria eurlex (stessa di 03_enrich_texts) — bypass del blocco JS."""
    celex = str(celex).strip()
    try:
        html = eurlex.get_html_by_celex_id(celex, language='it')
        # fallback inglese se italiano non disponibile
        if not html or len(html) < 500:
            html = eurlex.get_html_by_celex_id(celex, language='en')
        if not html or len(html) < 500:
            return None, 'not_found'
        STRUCTURAL_TAGS = ['eli-subdivision', 'oj-doc-ti', 'doc-ti']
        if not any(tag in html for tag in STRUCTURAL_TAGS):
            return None, 'no_structure'
        return html, 'ok'
    except Exception as e:
        err = str(e).lower()
        if 'timeout' in err:                   return None, 'timeout'
        if '404' in err or 'not found' in err: return None, 'not_found'
        return None, 'error'


def extract_title(soup):
    for css_class in ['oj-doc-ti', 'doc-ti']:
        tags = soup.find_all('p', class_=css_class)
        if not tags:
            continue
        parti = []
        for tag in tags:
            testo = tag.get_text(separator=' ', strip=True)
            if testo.startswith(('ANNEX', 'SCHEDULE', 'APPENDIX', 'ALLEGATO')):
                break
            parti.append(testo)
        if parti:
            return ' '.join(parti)[:MAX_TITLE_CHARS]
    title_tag = soup.find('title')
    if title_tag:
        text = re.sub(r'\s*[-–|]\s*EUR-Lex.*$', '',
                      title_tag.get_text(strip=True), flags=re.IGNORECASE)
        if len(text) > 20:
            return text[:MAX_TITLE_CHARS]
    return None


def extract_preamble_segments(soup):
    subdivisions = soup.find_all(class_='eli-subdivision')
    if not subdivisions:
        return []
    testo = re.sub(r'\s+', ' ',
                   subdivisions[0].get_text(separator=' ', strip=True)).strip()
    for marker in PREAMBLE_END_MARKERS:
        idx = testo.find(marker)
        if idx != -1:
            testo = testo[:idx].strip()
            break
    if len(testo) < 50:
        return []
    parts    = re.split(r'\((\d+)\)\s+', testo)
    segments = []
    if len(parts) <= 1:
        segments.append({'tipo': 'preambolo', 'identificatore': '0', 'testo': testo})
        return segments
    header = parts[0].strip()
    if header and len(header) >= 30:
        segments.append({'tipo': 'preambolo_header', 'identificatore': '0', 'testo': header})
    i = 1
    while i < len(parts) - 1:
        num, corpo = parts[i].strip(), parts[i + 1].strip()
        if corpo and len(corpo) >= 20:
            segments.append({'tipo': 'considerando', 'identificatore': num, 'testo': corpo})
        i += 2
    return segments


def extract_articles_full(soup):
    article_divs = []
    for div in soup.find_all(class_='eli-subdivision'):
        if re.match(r'^art_\d+', div.get('id', '')):
            article_divs.append(div)
    if not article_divs:
        article_divs = soup.find_all(class_='eli-subdivision',
                                     attrs={'data-section': 'article'})
    if not article_divs:
        for div in soup.find_all(class_='eli-subdivision')[1:]:
            if re.match(r'^Article\s+\d+',
                        div.get_text(separator=' ', strip=True)[:100], re.IGNORECASE):
                article_divs.append(div)
    segments = []
    for div in article_divs:
        testo = re.sub(r'\s+', ' ', div.get_text(separator=' ', strip=True)).strip()
        if len(testo) < 10:
            continue
        m   = re.match(r'Article\s+(\d+[a-z]?)', testo, re.IGNORECASE)
        num = m.group(1) if m else re.sub(r'^art_', '', div.get('id', str(len(segments)+1)))
        segments.append({'tipo': 'articolo', 'identificatore': num, 'testo': testo})
    return segments


def extract_and_parse(celex: str) -> dict:
    """Pipeline completa: fetch → parse → segmenti strutturati."""
    empty = {
        'titolo': celex, 'full_text': '', 'segments': '[]',
        'n_articoli': 0, 'n_considerando': 0, 'quality_score': 5,
    }
    html, status = fetch_eurlex_html_lib(celex)
    if status != 'ok':
        print(f"  [SKIP] status={status}")
        return empty

    soup  = BeautifulSoup(html, 'html.parser')
    title = extract_title(soup) or celex
    preamble_segs = extract_preamble_segments(soup)
    article_segs  = extract_articles_full(soup)

    all_segs = preamble_segs + article_segs
    for i, s in enumerate(all_segs):
        s['segment_id'] = i

    full_text = '\n\n'.join(s['testo'] for s in all_segs)
    n_art = sum(1 for s in all_segs if s['tipo'] == 'articolo')
    n_rec = sum(1 for s in all_segs if s['tipo'] == 'considerando')
    score = 90 if n_art >= 30 else 70 if n_art >= 10 else 40 if n_art >= 1 else 5

    return {
        'titolo':         title,
        'full_text':      full_text,
        'segments':       json.dumps(all_segs, ensure_ascii=False),
        'n_articoli':     n_art,
        'n_considerando': n_rec,
        'quality_score':  score,
    }


# ── 9.3  Loop principale  ────────────────────────────────────────────────────

def celex_to_layer(celex: str) -> str:
    code = celex[8] if len(celex) > 8 else '?'
    return {'L': 'Direttiva EU', 'R': 'Regolamento EU', 'D': 'Decisione EU'}.get(code, 'Atto EU')


rows = []
for celex in celex_ids:
    print(f"\n── {celex} ──────────────────────────────────────")
    parsed = extract_and_parse(celex)
    print(f"  Titolo:       {parsed['titolo'][:70]}")
    print(f"  articoli={parsed['n_articoli']}  considerando={parsed['n_considerando']}  score={parsed['quality_score']}")
    rows.append({
        'celex':          celex,
        'label':          celex,
        'titolo':         parsed['titolo'],
        'giurisdizione':  'EU',
        'layer_atteso':   celex_to_layer(celex),
        'fonte':          'eurlex_lib',
        'n_articoli':     parsed['n_articoli'],
        'n_considerando': parsed['n_considerando'],
        'quality_score':  parsed['quality_score'],
        'full_text':      parsed['full_text'],
        'segments':       parsed['segments'],
    })
    time.sleep(DELAY)


# ── 9.4  Salvataggio  ────────────────────────────────────────────────────────

df_eu = pd.DataFrame(rows)

# ── nodes_texts_eu_appalti.csv  (con full_text — per notebook 04) ─────────
eu_texts_path = OUT_DIR / 'nodes_texts_eu_appalti.csv'
df_eu.to_csv(eu_texts_path, index=False, encoding='utf-8')
print(f"✓ {eu_texts_path}")

# ── nodes_eu_appalti.csv  (solo metadati — per il grafo) ──────────────────
eu_nodes_path = OUT_DIR / 'nodes_eu_appalti.csv'
df_eu.drop(columns=['full_text', 'segments']).to_csv(eu_nodes_path, index=False, encoding='utf-8')
print(f"✓ {eu_nodes_path}")

# ── Cache globale (data/processed/nodes_texts.csv)  ───────────────────────
# Aggiorna la cache globale usata da 03_enrich_texts per evitare refetch.
# Formato compatibile: colonna Label = celex, + TEXT_COLS standard.
global_cache_path = Path('..') / 'data' / 'processed' / 'nodes_texts.csv'
global_cache_path.parent.mkdir(parents=True, exist_ok=True)

df_eu_cache = df_eu.rename(columns={'celex': 'Label'})[
    ['Label', 'titolo', 'full_text', 'segments']
].copy()
df_eu_cache['text_status']    = df_eu['quality_score'].apply(lambda s: 'ok' if s > 5 else 'no_content')
df_eu_cache['text_length']    = df_eu['full_text'].str.len().fillna(0).astype(int)
df_eu_cache['n_segments']     = df_eu['segments'].apply(
    lambda s: len(json.loads(s)) if pd.notna(s) and s != '[]' else 0
)
df_eu_cache['sections_found'] = 'title,preamble,articles'
df_eu_cache['preamble']       = None
df_eu_cache['articles']       = None
df_eu_cache['annexes']        = None
df_eu_cache['title']          = df_eu['titolo']

if global_cache_path.exists():
    existing = pd.read_csv(global_cache_path, low_memory=False)
    # aggiungi solo i CELEX non già presenti
    new_only = df_eu_cache[~df_eu_cache['Label'].isin(existing['Label'])]
    if len(new_only) > 0:
        updated = pd.concat([existing, new_only], ignore_index=True)
        updated.to_csv(global_cache_path, index=False, encoding='utf-8')
        print(f"✓ Cache globale aggiornata: +{len(new_only)} CELEX EU  ({global_cache_path})")
    else:
        print(f"  Cache globale: tutti i CELEX EU già presenti, nessuna modifica.")
else:
    df_eu_cache.to_csv(global_cache_path, index=False, encoding='utf-8')
    print(f"✓ Cache globale creata: {len(df_eu_cache)} CELEX EU  ({global_cache_path})")

# ── Riepilogo ─────────────────────────────────────────────────────────────
print("\n── Riepilogo EU ─────────────────────────────────────────────────────────")
for _, r in df_eu.iterrows():
    print(f"  {r['celex']:15s}  {r['layer_atteso']:15s}  "
          f"art={r['n_articoli']:3d}  rec={r['n_considerando']:3d}  score={r['quality_score']}")

CELEX da recuperare: 6

── 32004L0017 ──────────────────────────────────────
  [SKIP] status=no_structure
  Titolo:       32004L0017
  articoli=0  considerando=0  score=5

── 32004L0018 ──────────────────────────────────────
  Titolo:       DIRETTIVA 2004/18/CE DEL PARLAMENTO EUROPEO E DEL CONSIGLIO del 31 mar
  articoli=0  considerando=0  score=5

── 32006L0123 ──────────────────────────────────────
  Titolo:       DIRETTIVA 2006/123/CE DEL PARLAMENTO EUROPEO E DEL CONSIGLIO del 12 di
  articoli=46  considerando=118  score=90

── 32014L0023 ──────────────────────────────────────
  Titolo:       DIRETTIVA 2014/23/UE DEL PARLAMENTO EUROPEO E DEL CONSIGLIO del 26 feb
  articoli=55  considerando=88  score=90

── 32014L0024 ──────────────────────────────────────
  Titolo:       DIRETTIVA 2014/24/UE DEL PARLAMENTO EUROPEO E DEL CONSIGLIO del 26 feb
  articoli=94  considerando=138  score=90

── 32014L0025 ──────────────────────────────────────
  Titolo:       DIRETTIVA 2014/25/UE DEL PARLAME